In [1]:
import json
import torch
import numpy as np
from transformers import AutoTokenizer
import torch.nn as nn
from transformers import AutoModel


In [2]:
class LabelWiseAttention(nn.Module):
    def __init__(self, hidden_size, num_labels):
        super(LabelWiseAttention, self).__init__()
        self.label_query = nn.Parameter(torch.randn(num_labels, hidden_size))
        self.linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, encoder_output, attention_mask):
        Q = self.label_query
        K = self.linear(encoder_output)
        attention_scores = torch.matmul(K, Q.t())
        attention_scores = attention_scores.masked_fill(
            attention_mask.unsqueeze(-1) == 0, float('-inf')
        )
        attention_weights = torch.softmax(attention_scores, dim=1)
        weighted_sum = torch.einsum("bsl,bsh->blh", attention_weights, encoder_output)
        return weighted_sum


class LabelWiseAttentionBinaryPolarityClassifier(nn.Module):
    def __init__(self, model_name, n_mech_labels):
        super(LabelWiseAttentionBinaryPolarityClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Label-wise attention mechanism head
        self.attention = LabelWiseAttention(hidden_size, n_mech_labels)
        self.mech_classifier = nn.Linear(hidden_size, 1)  # one per label

        # 3-class polarity head (positive vs. negative vs. neutral)
        self.polarity_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 3)  # 3 classes
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        encoder_output = outputs.last_hidden_state

        # Mechanism predictions
        attended = self.attention(encoder_output, attention_mask)  # [B, L, H]
        mech_logits = self.mech_classifier(attended).squeeze(-1)   # [B, L]

        # Polarity prediction (use CLS token)
        cls_output = encoder_output[:, 0, :]
        polarity_logits = self.polarity_head(cls_output)    # [B, 3]

        return mech_logits, polarity_logits


In [3]:



# -------- Device --------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- Load Supporting Files --------
with open("/kaggle/input/output-2/label_columns.json") as f:
    label_columns = json.load(f)

with open("/kaggle/input/output-2/polarity_labels.json") as f:
    polarity_classes = json.load(f)

with open("/kaggle/input/output-2/test_thresholds_batch_1.json") as f:
    thresholds = json.load(f)

# -------- Rebuild MultiLabelBinarizer --------
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=label_columns)
mlb.fit([[]])  # Fit on empty list just to load correct classes

# -------- Load Tokenizer & Model --------
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

model = LabelWiseAttentionBinaryPolarityClassifier(
    model_name="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    n_mech_labels=len(label_columns)
)
model.load_state_dict(torch.load("/kaggle/input/output/test_model_batch_1.pt", map_location=device))
model.to(device)
model.eval()

# -------- Rebuild Polarity Encoder --------
from sklearn.preprocessing import LabelEncoder
polarity_encoder = LabelEncoder()
polarity_encoder.classes_ = np.array(polarity_classes)

# -------- Test Examples --------
implicit_examples = [
        # Format: (text, mechanism_label, polarity_label)
        ("The transcription factor binds to its own promoter region.", "autoregulation", "neutral"),
        ("The enzyme activates itself through conformational change.", "autoactivation", "positive"),
        ("The protein phosphorylates itself on a tyrosine residue.", "autophosphorylation", "positive"),
        ("The protease cleaves itself to generate the active form.", "autocatalysis", "positive"),
        ("The cell produces molecules that signal itself to change behavior.", "autoinduction", "positive"),
        ("The receptor signals to reduce its own expression level.", "autoinhibition", "negative"),
        ("Upon binding ligand, the receptor undergoes a conformational change that enables phosphorylation of its cytoplasmic domain.", "autophosphorylation", "positive"),
        ("The transcription factor negatively controls expression of its own gene.", "autoregulation", "negative"),
        ("The kinase domain transfers phosphate groups to residues within the same protein.", "autophosphorylation", "positive"),
        ("This bacterial system uses cell-to-cell signaling to coordinate population behavior.", "autoinduction", "positive"),
        ("The peptide recognizes and binds specifically to the same protein it was derived from.", "autofeedback", "neutral"),
        ("The dimeric protein activates by cross-phosphorylation between the two identical subunits.", "autoactivation", "positive"),
        ("AGPCRs uniquely contain large, self-proteolyzing extracellular regions.", "autocatalysis", "positive"),
        ("GAIN domain-mediated self-cleavage is constitutive and produces two-fragment holoreceptors.", "autocatalysis", "positive"),
        ("The self-repression function of IbpA is conserved in other γ-proteobacterial IbpAs.", "autoinhibition", "negative"),
        ("A cationic residue-rich region is critical for the self-suppression activity.", "autoinhibition", "negative"),
        ("We propose a negative feedback loop, in which sphingosine inhibits GBA2 activity.", "autoinhibition", "negative"),
        ("DNA damage-induced activation of p53 initiates a negative-feedback loop which rapidly downregulates RAG1 levels.", "autoregulation", "negative")
    ]

# -------- Run Predictions at Multiple Thresholds --------
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for text, true_mech, true_pol in implicit_examples:
    # Tokenize
    encoding = tokenizer(text, padding='max_length', truncation=True, max_length=256, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Forward pass
    with torch.no_grad():
        mech_logits, pol_logits = model(input_ids, attention_mask)

    # Polarity decoding (unchanged)
    pol_pred_idx = torch.argmax(pol_logits, dim=1).item()
    pol_pred_label = polarity_encoder.classes_[pol_pred_idx]

    print("📘 TEXT:", text)
    print("🔹 True Mechanism:", true_mech)
    print("🔹 True Polarity:", true_pol)
    print("🔸 Predicted Polarity:", pol_pred_label)

    # Decode mechanisms for each threshold
    mech_probs = torch.sigmoid(mech_logits).squeeze(0).cpu().numpy()
    for thresh in thresholds_to_test:
        mech_preds = [mlb.classes_[i] for i, p in enumerate(mech_probs) if p >= thresh]
        print(f"    🔸 Threshold {thresh:.1f} → Predicted Mechanisms: {mech_preds}")

    print("------")


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

2025-05-28 05:47:58.532983: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748411278.739495      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748411278.801203      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

📘 TEXT: The transcription factor binds to its own promoter region.
🔹 True Mechanism: autoregulation
🔹 True Polarity: neutral
🔸 Predicted Polarity: neutral
    🔸 Threshold 0.3 → Predicted Mechanisms: []
    🔸 Threshold 0.4 → Predicted Mechanisms: []
    🔸 Threshold 0.5 → Predicted Mechanisms: []
    🔸 Threshold 0.6 → Predicted Mechanisms: []
    🔸 Threshold 0.7 → Predicted Mechanisms: []
    🔸 Threshold 0.8 → Predicted Mechanisms: []
------
📘 TEXT: The enzyme activates itself through conformational change.
🔹 True Mechanism: autoactivation
🔹 True Polarity: positive
🔸 Predicted Polarity: negative
    🔸 Threshold 0.3 → Predicted Mechanisms: ['autophosphorylation']
    🔸 Threshold 0.4 → Predicted Mechanisms: ['autophosphorylation']
    🔸 Threshold 0.5 → Predicted Mechanisms: []
    🔸 Threshold 0.6 → Predicted Mechanisms: []
    🔸 Threshold 0.7 → Predicted Mechanisms: []
    🔸 Threshold 0.8 → Predicted Mechanisms: []
------
📘 TEXT: The protein phosphorylates itself on a tyrosine residue.
🔹 T

In [4]:
print("📈 Mechanism Probabilities:")
for i, mech in enumerate(label_columns):
    avg_prob = np.mean([torch.sigmoid(mech_logits).squeeze(0).cpu().numpy()[i] for text, _, _ in implicit_examples])
    print(f"{mech}: {avg_prob:.4f}")


📈 Mechanism Probabilities:
autoactivation: 0.0440
autocatalysis: 0.0572
autofeedback: 0.0404
autoinduction: 0.0472
autoinhibition: 0.0713
autokinase: 0.0774
autolysis: 0.2055
autophosphorylation: 0.1112
autoregulation: 0.1209
autoubiquitination: 0.0530


In [5]:
# -------- Predict Polarity Only --------
for text, true_mech, true_pol in implicit_examples:
    # Tokenize input
    encoding = tokenizer(text, padding='max_length', truncation=True, max_length=256, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Forward pass
    with torch.no_grad():
        _, pol_logits = model(input_ids, attention_mask)  # Ignore mech_logits

    # Decode polarity prediction
    pol_pred_idx = torch.argmax(pol_logits, dim=1).item()
    pol_pred_label = polarity_encoder.classes_[pol_pred_idx]

    # Output
    print("📘 TEXT:", text)
    print("🔹 True Polarity:", true_pol)
    print("🔸 Predicted Polarity:", pol_pred_label)
    print("------")


📘 TEXT: The transcription factor binds to its own promoter region.
🔹 True Polarity: neutral
🔸 Predicted Polarity: neutral
------
📘 TEXT: The enzyme activates itself through conformational change.
🔹 True Polarity: positive
🔸 Predicted Polarity: negative
------
📘 TEXT: The protein phosphorylates itself on a tyrosine residue.
🔹 True Polarity: positive
🔸 Predicted Polarity: positive
------
📘 TEXT: The protease cleaves itself to generate the active form.
🔹 True Polarity: positive
🔸 Predicted Polarity: negative
------
📘 TEXT: The cell produces molecules that signal itself to change behavior.
🔹 True Polarity: positive
🔸 Predicted Polarity: neutral
------
📘 TEXT: The receptor signals to reduce its own expression level.
🔹 True Polarity: negative
🔸 Predicted Polarity: neutral
------
📘 TEXT: Upon binding ligand, the receptor undergoes a conformational change that enables phosphorylation of its cytoplasmic domain.
🔹 True Polarity: positive
🔸 Predicted Polarity: positive
------
📘 TEXT: The transcri